In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.signal import butter, filtfilt, convolve, welch

# --- CONFIGURACIÓN Y CONSTANTES ---
FS = 1259 # Frecuencia de muestreo (Hz)
LOW_CUT = 20.0
HIGH_CUT = 450.0
WINDOW_SIZE_MS = 100 
ACTIVATION_THRESHOLD = 10 # Umbral de activación (10%)

TRIAL_FILES = ['trial_5.csv', 'trial_6.csv', 'trial_9.csv', 'trial_10.csv', 'trial_11.csv', 'trial_12.csv']

# Mapeo de columnas de EMG para los archivos de trial
COLUMN_MAP_TRIAL = {
    'trial_5.csv': 'R BICEPS BRACHII: EMG 1 [V]', 
    'trial_6.csv': 'L BICEPS BRACHII: EMG 4 [V]',
    'trial_9.csv': 'R DELTOID ANTERIOR: EMG 2 [V]',
    'trial_10.csv': 'L DELTOID ANTERIOR: EMG 3 [V]',
    'trial_11.csv': 'R DELTOID POSTERIOR: EMG 7 [V]',
    'trial_12.csv': 'L DELTOID POSTERIOR: EMG 5 [V]'
}

# Mapeo de nombres de músculo y nombres de archivo MVC esperado
# (Nombre corto para display, Nombre completo para lógica de lado, Nombre de columna EMG)
MUSCLE_MAP = {
    # Para BICEPS, la parte 'BRACHII' se omite y el lado (R/L) puede o no estar en el nombre del archivo MVC.
    'trial_5.csv': ('R BICEPS', 'R_BICEPS', COLUMN_MAP_TRIAL['trial_5.csv']),
    'trial_6.csv': ('L BICEPS', 'L_BICEPS', COLUMN_MAP_TRIAL['trial_6.csv']),
    # Para DELTOID, se mantiene la estructura
    'trial_9.csv': ('R DELT ANT', 'R_DELTOID_ANTERIOR', COLUMN_MAP_TRIAL['trial_9.csv']),
    'trial_10.csv': ('L DELT ANT', 'L_DELTOID_ANTERIOR', COLUMN_MAP_TRIAL['trial_10.csv']),
    'trial_11.csv': ('R DELT POST', 'R_DELTOID_POSTERIOR', COLUMN_MAP_TRIAL['trial_11.csv']),
    'trial_12.csv': ('L DELT POST', 'L_DELTOID_POSTERIOR', COLUMN_MAP_TRIAL['trial_12.csv'])
}

# --- FUNCIONES DE FILTRADO Y MÉTRICAS ---

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return b, a

def bandpass_filter(data, lowcut, highcut, fs):
    if data.size == 0: return np.array([])
    b, a = butter_bandpass(lowcut, highcut, fs)
    try:
        return filtfilt(b, a, data)
    except ValueError:
        return np.array([])

def rms(signal):
    # Devuelve np.nan si la señal está vacía o es esencialmente cero
    if signal.size == 0 or np.allclose(signal, 0): 
        return np.nan
    return np.sqrt(np.mean(signal**2))

def iemg(signal):
    return np.trapz(np.abs(signal)) if signal.size > 0 else np.nan

def calculate_psd(signal, fs):
    """ CORRECCIÓN: nperseg dinámico para evitar advertencias. """
    if signal.size == 0: return np.array([]), np.array([])
    nperseg_val = min(signal.size, int(fs)) 
    f, Pxx = welch(signal, fs, window='hann', nperseg=nperseg_val, scaling='density')
    return f, Pxx

def mnf(signal, fs):
    f, Pxx = calculate_psd(signal, fs)
    if Pxx.size == 0 or np.sum(Pxx) == 0: return np.nan
    return np.sum(f * Pxx) / np.sum(Pxx)

def mdf(signal, fs):
    f, Pxx = calculate_psd(signal, fs)
    if Pxx.size == 0 or np.sum(Pxx) == 0: return np.nan
    total_power = np.sum(Pxx)
    cumsum = np.cumsum(Pxx)
    median_index = np.where(cumsum >= total_power * 0.5)[0]
    return f[median_index[0]] if median_index.size > 0 else np.nan

# --- FUNCIONES DE SEGMENTACIÓN Y EXTRACCIÓN ---

def segment_emg_by_rms_envelope(filtered_signal, fs, window_size_ms, threshold_percent):
    window_size_samples = int(fs * (window_size_ms / 1000.0))
    if window_size_samples == 0 or len(filtered_signal) < window_size_samples:
        return []
    
    rms_envelope = np.sqrt(convolve(filtered_signal**2, np.ones(window_size_samples)/window_size_samples, mode='valid'))
    if rms_envelope.size == 0: return []
    
    max_rms_env = np.max(rms_envelope)
    
    if max_rms_env < 1e-6:
        return []
    
    threshold = max_rms_env * (threshold_percent / 100.0)
    
    is_active = rms_envelope > threshold
    diff_active = np.diff(np.pad(is_active, (1, 0), 'constant', constant_values=0).astype(int))

    starts_env = np.where(diff_active == 1)[0]
    stops_env = np.where(diff_active == -1)[0]
    
    offset = window_size_samples - 1 
    starts = [s + offset for s in starts_env]
    stops = [s + offset for s in stops_env]

    if stops and starts and stops[0] < starts[0]: starts = starts[1:]
    if starts and stops and starts[-1] > stops[-1]: stops = stops[:-1]

    segments = []
    min_segment_len = 100 
    for start, stop in zip(starts, stops):
        if stop > start and (stop - start) > min_segment_len: 
            segments.append(filtered_signal[start:stop])

    return segments

def extract_features_per_segment(emg_segments, fs):
    segment_features = []
    for segment in emg_segments:
        segment_features.append([rms(segment), iemg(segment), mnf(segment, fs), mdf(segment, fs)])
    
    if len(segment_features) == 0:
        return np.array([np.nan, np.nan, np.nan, np.nan])
        
    return np.nanmean(segment_features, axis=0)

# --- FUNCIÓN PARA EXTRAER RMS_MVC (CORREGIDA) ---

def get_mvc_rms(subject_id, muscle_name_full, column_name_trial):
    """
    Calcula el RMS máximo (MVC) a partir del archivo MVC específico.
    Ajusta la convención de nombres de archivo MVC (ej: BICEPS vs R_BICEPS_BRACHII).
    """
    
    # 1. Determinar el nombre base del músculo para el archivo MVC
    if 'BICEPS' in muscle_name_full:
        # Si es BICEPS, el nombre del archivo puede ser solo BICEPS (sin lado R/L) o L_BICEPS/R_BICEPS
        # Intentamos primero con el nombre completo simplificado (ej: R_BICEPS)
        if 'R_BICEPS' in muscle_name_full:
             mvc_muscle_base = 'R_BICEPS'
        elif 'L_BICEPS' in muscle_name_full:
             mvc_muscle_base = 'L_BICEPS'
        else: # Si no tiene lado explícito en el mapeo, usamos BICEPS
             mvc_muscle_base = 'BICEPS'
             
    elif 'DELTOID' in muscle_name_full:
        # Para DELTOID, asumimos la convención completa (ej: R_DELTOID_ANTERIOR)
        mvc_muscle_base = muscle_name_full 
    else:
        # Fallback si no se reconoce
        mvc_muscle_base = muscle_name_full

    # 2. Construcción de la ruta del archivo MVC
    muscle_file_name = f'subject_{subject_id}_MVC_{mvc_muscle_base}.csv'
    mvc_path = os.path.join(f'sEMG_data/subject_{subject_id}/MVC', muscle_file_name)
    
    # 3. Intento de lectura y procesamiento
    if not os.path.exists(mvc_path):
        # Si la convención R/L_BICEPS falla, intentamos la convención BICEPS a secas
        if 'BICEPS' in mvc_muscle_base:
            muscle_file_name_alt = f'subject_{subject_id}_MVC_BICEPS.csv'
            mvc_path_alt = os.path.join(f'sEMG_data/subject_{subject_id}/MVC', muscle_file_name_alt)
            if os.path.exists(mvc_path_alt):
                mvc_path = mvc_path_alt
            else:
                # Si ambos fallan
                print(f"Advertencia: Archivo MVC no encontrado para Sujeto_{subject_id}, Músculo: {mvc_muscle_base} en {mvc_path} ni en {mvc_path_alt}")
                return np.nan
        else:
            print(f"Advertencia: Archivo MVC no encontrado para Sujeto_{subject_id}, Músculo: {mvc_muscle_base} en {mvc_path}")
            return np.nan

    try:
        mvc_data = pd.read_csv(mvc_path, delimiter=',', header=0)
        
        # Buscar la columna EMG en el archivo MVC. Buscamos por la parte del nombre del músculo
        muscle_name_prefix = column_name_trial.split(':')[0].strip()
        valid_cols = [col for col in mvc_data.columns if muscle_name_prefix in col and '[V]' in col and 'filter' not in col]
        
        if not valid_cols:
            print(f"Error: Columna EMG no encontrada en el archivo MVC: {mvc_path}")
            return np.nan
            
        emg_signal_mvc = mvc_data[valid_cols[0]].dropna().values

        if emg_signal_mvc.size == 0:
            return np.nan
        
        # Filtrar la señal MVC
        filtered_signal_mvc = bandpass_filter(emg_signal_mvc, LOW_CUT, HIGH_CUT, FS)
        
        # El MVC es el valor máximo (pico) de la señal EMG filtrada y rectificada
        # Es la práctica estándar usar el valor pico de la señal rectificada.
        return np.max(np.abs(filtered_signal_mvc))
        
    except Exception as e:
        print(f"Error al procesar el archivo MVC: {mvc_path}. Error: {e}")
        return np.nan

# --- FUNCIÓN PRINCIPAL DE PROCESAMIENTO POR SUJETO ---

def process_subject(subject_id, mvc_cache):
    subject_dir = f'sEMG_data/subject_{subject_id}'
    subject_results = []
    
    # 1. Obtener y cachear todos los MVC para el sujeto
    if subject_id not in mvc_cache:
        mvc_cache[subject_id] = {}
        for file_name, (short_name, full_name, col_name) in MUSCLE_MAP.items():
            mvc_value = get_mvc_rms(subject_id, full_name, col_name)
            mvc_cache[subject_id][file_name] = mvc_value

    for file_name in TRIAL_FILES:
        file_path = os.path.join(subject_dir, file_name)
        short_name, full_name, emg_col_to_analyze = MUSCLE_MAP.get(file_name)
        
        # Inicializar características a NaN
        rms_norm_avg = np.nan
        iemg_avg = np.nan
        mnf_avg = np.nan
        mdf_avg = np.nan
        
        # 1. Obtener el RMS_MVC (normalizador)
        mvc_rms = mvc_cache[subject_id].get(file_name, np.nan)
        
        if not os.path.exists(file_path) or emg_col_to_analyze is None:
            # Archivo de trial faltante
            pass
        else:
            try:
                data = pd.read_csv(file_path, delimiter=',', header=0)
                emg_signal = data[emg_col_to_analyze].dropna().values
                
                if emg_signal.size > 0:
                    filtered_signal = bandpass_filter(emg_signal, LOW_CUT, HIGH_CUT, FS)
                    
                    # Segmentación y extracción de características ABSOLUTAS
                    emg_segments = segment_emg_by_rms_envelope(filtered_signal, FS, WINDOW_SIZE_MS, ACTIVATION_THRESHOLD)
                    avg_features_abs = extract_features_per_segment(emg_segments, FS)
                    
                    rms_abs_avg = avg_features_abs[0]
                    iemg_avg = avg_features_abs[1]
                    mnf_avg = avg_features_abs[2]
                    mdf_avg = avg_features_abs[3]
                    
                    # 2. NORMALIZACIÓN RMS (si MVC está disponible)
                    if not np.isnan(rms_abs_avg) and not np.isnan(mvc_rms) and mvc_rms > 0:
                        # Se normaliza el RMS promedio del trial entre el MVC máximo (pico)
                        rms_norm_avg = (rms_abs_avg / mvc_rms) * 100
                    
            except Exception as e:
                # print(f"Error al procesar trial {file_name} de Sujeto_{subject_id}: {e}")
                pass
        
        # Crear la fila de resultados para el DataFrame
        row = [f'Subject_{subject_id}', file_name, rms_norm_avg, iemg_avg, mnf_avg, mdf_avg]
        subject_results.append(row)

    # Convertir a DataFrame
    # Se ajusta el nombre de la columna RMS para reflejar la normalización
    df = pd.DataFrame(subject_results, columns=['Subject', 'Trial', 'RMS (Avg) [% MVC]', 'iEMG (Avg) [V.s]', 'MNF (Avg) [Hz]', 'MDF (Avg) [Hz]'])
    return df

# --- EJECUCIÓN MAESTRA ---

ALL_SUBJECTS_DATA = []
NUM_SUBJECTS = 13
subject_1_df = None
mvc_cache = {} # Cache para guardar los valores MVC y evitar re-lectura

print("Iniciando procesamiento de datos y normalización MVC...")

for i in range(1, NUM_SUBJECTS + 1):
    subject_df = process_subject(i, mvc_cache)
    ALL_SUBJECTS_DATA.append(subject_df)
    
    if i == 1:
        subject_1_df = subject_df
        
# 1. Concatenar todos los resultados en un solo DataFrame
final_df = pd.concat(ALL_SUBJECTS_DATA, ignore_index=True)

# 2. Guardar el archivo CSV con todos los resultados
final_df.to_csv("EMG_All_Subjects_Features_Normalized.csv", index=False)


# 3. Mostrar la tabla de resultados para Subject_1 (Verificación)
if subject_1_df is not None:
    # Mapeo actualizado para la visualización
    muscle_map_display = {k: v[0] for k, v in MUSCLE_MAP.items()}
    
    # Reformatear para la visualización en pantalla
    display_df = subject_1_df.copy().set_index('Trial').drop('Subject', axis=1)
    display_df.index = [f'{idx} ({muscle_map_display.get(idx)})' for idx in display_df.index]
    
    print("\n" + "="*100)
    print(" RESULTADOS FINALES: Características Promedio Normalizadas (Sujeto 1 - Verificación)")
    print("  (Todos los resultados han sido guardados en 'EMG_All_Subjects_Features_Normalized.csv')")
    print("="*100)
    print(display_df.to_string(float_format="{:.4f}".format)) 
    print("="*100)

print(f"\nArchivo 'EMG_All_Subjects_Features_Normalized.csv' creado con los datos de {NUM_SUBJECTS} sujetos.")